# 🛰️ Multimodal Remote Sensing Land Classification

---

| **Course - B.Tech.** | **Semester – 4th** |
|---|---|
| **Course Code - CSET301** | **Course Name - AIML** |
| **Year – 2025** | **Semester - Even** |

**PROJECT NOTEBOOK**

**Title: Multimodal Remote Sensing Land Classification using SAR + Optical Imagery (EuroSAT Dataset) with ResNet-18**

---

## 🎯 Objective
To implement a **Multimodal Deep Learning** pipeline for land use/land cover classification using both **SAR (Synthetic Aperture Radar)** and **Optical (RGB)** image patches from the **EuroSAT** dataset.

We train **three models**:
1. **Model A** – ResNet-18 on Optical (RGB) images only
2. **Model B** – ResNet-18 on SAR (simulated single-channel) images only
3. **Model C** – Feature Fusion Model combining SAR + Optical features

Each model is trained on **100 image patches per class** and compared using accuracy and F1-score.

### 📌 Dataset: EuroSAT (Sentinel-2 Patches)
- **Source:** TorchGeo / [EuroSAT GitHub](https://github.com/phelber/EuroSAT)
- **Image size:** 64×64 pixels
- **Classes:** 10 land-use categories
- **Used:** 100 patches per class (1000 total)
- **Modalities:** RGB optical + SAR (simulated from NIR/SWIR bands)

### 🗂️ Land Classes
| # | Class | Description |
|---|---|---|
| 0 | AnnualCrop | Annual agricultural fields |
| 1 | Forest | Dense tree cover |
| 2 | HerbaceousVegetation | Grasslands and shrubs |
| 3 | Highway | Roads and highways |
| 4 | Industrial | Factories and warehouses |
| 5 | Pasture | Open grazing land |
| 6 | PermanentCrop | Orchards/vineyards |
| 7 | Residential | Urban housing areas |
| 8 | River | Water bodies – rivers |
| 9 | SeaLake | Sea and lake water bodies |


---
## 📦 Step 1: Install Dependencies

In [ ]:
# Install required libraries
!pip install torchgeo timm scikit-learn matplotlib seaborn rasterio -q

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {DEVICE}')
print(f'   PyTorch version: {torch.__version__}')


---
## 📥 Step 2: Download EuroSAT Dataset (100 Patches per Class)

In [ ]:
import torchgeo.datasets as geo_datasets

# ── Configuration ──────────────────────────────────────────────────────────────
PATCHES_PER_CLASS = 100   # 100 patches per class
IMG_SIZE          = 64
NUM_CLASSES       = 10

CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
    'River', 'SeaLake'
]

# ── Load EuroSAT RGB (Optical) ─────────────────────────────────────────────────
print('📡 Loading EuroSAT RGB (Optical) dataset...')
full_dataset_rgb = geo_datasets.EuroSAT(
    root='./data', bands=['B04', 'B03', 'B02'], download=True)
print(f'   Total samples: {len(full_dataset_rgb)}')

# ── Load EuroSAT SAR (using NIR + SWIR bands to simulate SAR) ─────────────────
# Sentinel-2 has no native SAR, so we simulate SAR using
# bands B08 (NIR), B11 (SWIR1), B12 (SWIR2) which highlight
# structural/texture features similar to SAR backscatter.
print('\n📡 Loading EuroSAT SAR-simulated dataset (NIR+SWIR bands)...')
full_dataset_sar = geo_datasets.EuroSAT(
    root='./data', bands=['B08', 'B11', 'B12'], download=True)
print(f'   Total samples: {len(full_dataset_sar)}')

# ── Sample 100 patches per class for both modalities ──────────────────────────
print(f'\n🔪 Sampling {PATCHES_PER_CLASS} patches per class...')

opt_images, sar_images, labels_all = [], [], []
class_counts = {i: 0 for i in range(NUM_CLASSES)}

all_indices = list(range(len(full_dataset_rgb)))
random.shuffle(all_indices)

def process_patch(sample, clip_val=3000):
    img_np = sample['image'].permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, clip_val)
    img_np = (img_np / clip_val * 255).astype(np.uint8)
    return Image.fromarray(img_np)

for idx in all_indices:
    sample_rgb = full_dataset_rgb[idx]
    label      = sample_rgb['label'].item()

    if class_counts[label] < PATCHES_PER_CLASS:
        sample_sar = full_dataset_sar[idx]
        opt_images.append(process_patch(sample_rgb))
        sar_images.append(process_patch(sample_sar))
        labels_all.append(label)
        class_counts[label] += 1

    if all(v == PATCHES_PER_CLASS for v in class_counts.values()):
        break

print(f'\n✅ Sampling complete!')
print(f'   Optical patches : {len(opt_images)}')
print(f'   SAR patches     : {len(sar_images)}')
print(f'   Labels          : {len(labels_all)}')


---
## 🔍 Step 3: Data Exploration & Visualization

In [ ]:
# ── Dataset Summary ────────────────────────────────────────────────────────────
print('='*60)
print('          DATASET SUMMARY')
print('='*60)
print(f'  Total patches       : {len(labels_all)}')
print(f'  Patches per class   : {PATCHES_PER_CLASS}')
print(f'  Number of classes   : {NUM_CLASSES}')
print(f'  Image size          : {IMG_SIZE} × {IMG_SIZE} px')
print(f'  Modality 1 (Optical): RGB – Bands B4, B3, B2 (Sentinel-2)')
print(f'  Modality 2 (SAR sim): NIR/SWIR – Bands B8, B11, B12')
print('='*60)

sample_arr = np.array(opt_images[0])
print(f'\n  Optical image shape : {sample_arr.shape}')
print(f'  Pixel dtype         : {sample_arr.dtype}')
print(f'  Pixel value range   : [{sample_arr.min()}, {sample_arr.max()}]')


In [ ]:
# ── Visualize Optical vs SAR Patches Side by Side ──────────────────────────────
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
fig.suptitle('EuroSAT – Optical (RGB) vs SAR-simulated (NIR/SWIR) per Class',
             fontsize=14, fontweight='bold', y=1.01)

shown = {i: False for i in range(NUM_CLASSES)}
opt_row = axes[:2].flatten()
sar_row = axes[2:].flatten()

for img_opt, img_sar, lbl in zip(opt_images, sar_images, labels_all):
    if not shown[lbl]:
        # Optical
        opt_row[lbl].imshow(img_opt)
        opt_row[lbl].set_title(f'[OPT] {CLASS_NAMES[lbl]}', fontsize=9, fontweight='bold')
        opt_row[lbl].axis('off')
        # SAR
        sar_row[lbl].imshow(img_sar)
        sar_row[lbl].set_title(f'[SAR] {CLASS_NAMES[lbl]}', fontsize=9, fontweight='bold',
                                color='darkred')
        sar_row[lbl].axis('off')
        shown[lbl] = True
    if all(shown.values()):
        break

plt.tight_layout()
plt.savefig('optical_vs_sar_patches.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Optical vs SAR visualization saved')


In [ ]:
# ── Class Distribution ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))
bars = ax.bar(CLASS_NAMES, [PATCHES_PER_CLASS]*NUM_CLASSES,
              color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Land Cover Class', fontsize=12)
ax.set_ylabel('Number of Patches', fontsize=12)
ax.set_title('Class Distribution – 100 Patches per Class (Balanced) | Both Modalities',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 130)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class distribution plot saved')


---
## 🔧 Step 4: Data Preparation & Custom Datasets

In [ ]:
# ── Normalization (ImageNet stats for ResNet-18 pretrained) ───────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ── Transforms ─────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print('✅ Transforms defined:')
print('   Train: Resize → Flip → Rotate → ColorJitter → Normalize')
print('   Val  : Resize → Normalize')


# ── Single-Modality Dataset ────────────────────────────────────────────────────
class SingleModalDataset(Dataset):
    """Dataset for one modality (optical OR SAR)."""
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        if img.mode != 'RGB':
            img = img.convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# ── Multimodal (Fusion) Dataset ────────────────────────────────────────────────
class MultimodalDataset(Dataset):
    """Dataset that returns BOTH optical and SAR patches for fusion model."""
    def __init__(self, opt_images, sar_images, labels, transform=None):
        self.opt_images = opt_images
        self.sar_images = sar_images
        self.labels     = labels
        self.transform  = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        opt = self.opt_images[idx]
        sar = self.sar_images[idx]
        if opt.mode != 'RGB': opt = opt.convert('RGB')
        if sar.mode != 'RGB': sar = sar.convert('RGB')
        if self.transform:
            opt = self.transform(opt)
            sar = self.transform(sar)
        return opt, sar, self.labels[idx]


# ── Train / Val / Test Split (60 / 20 / 20) ────────────────────────────────────
indices = list(range(len(labels_all)))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.4, stratify=labels_all, random_state=SEED)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5,
    stratify=[labels_all[i] for i in temp_idx], random_state=SEED)

train_opt = [opt_images[i] for i in train_idx]
val_opt   = [opt_images[i] for i in val_idx]
test_opt  = [opt_images[i] for i in test_idx]

train_sar = [sar_images[i] for i in train_idx]
val_sar   = [sar_images[i] for i in val_idx]
test_sar  = [sar_images[i] for i in test_idx]

train_lbl = [labels_all[i] for i in train_idx]
val_lbl   = [labels_all[i] for i in val_idx]
test_lbl  = [labels_all[i] for i in test_idx]

# ── DataLoaders ────────────────────────────────────────────────────────────────
BATCH_SIZE = 32

# Optical-only
train_loader_opt = DataLoader(
    SingleModalDataset(train_opt, train_lbl, train_transform),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader_opt   = DataLoader(
    SingleModalDataset(val_opt, val_lbl, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_opt  = DataLoader(
    SingleModalDataset(test_opt, test_lbl, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# SAR-only
train_loader_sar = DataLoader(
    SingleModalDataset(train_sar, train_lbl, train_transform),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader_sar   = DataLoader(
    SingleModalDataset(val_sar, val_lbl, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_sar  = DataLoader(
    SingleModalDataset(test_sar, test_lbl, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Fusion (Multimodal)
train_loader_mm = DataLoader(
    MultimodalDataset(train_opt, train_sar, train_lbl, train_transform),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader_mm   = DataLoader(
    MultimodalDataset(val_opt, val_sar, val_lbl, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_mm  = DataLoader(
    MultimodalDataset(test_opt, test_sar, test_lbl, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'\n✅ DataLoaders ready:')
print(f'   Train : {len(train_idx)} samples  |  Val: {len(val_idx)}  |  Test: {len(test_idx)}')
print(f'   Batch size: {BATCH_SIZE}')
print(f'   Loaders created: Optical-only, SAR-only, Fusion (Multimodal)')


---
## 🧠 Step 5: Model Architecture

We implement **three models** using **ResNet-18**:

| Model | Input | Description |
|---|---|---|
| **Model A** | Optical (RGB) | ResNet-18 pretrained, fine-tuned on optical patches |
| **Model B** | SAR (NIR/SWIR) | ResNet-18 pretrained, fine-tuned on SAR patches |
| **Model C** | Optical + SAR | Feature Fusion – dual ResNet-18 encoders, concat + classifier |


In [ ]:
# ── Model A & B: Single-modality ResNet-18 ────────────────────────────────────

def build_resnet18(num_classes=10, freeze_backbone=False):
    """
    ResNet-18 pretrained on ImageNet.
    Final FC replaced for land classification (512 → 256 → num_classes).
    """
    model = models.resnet18(weights='IMAGENET1K_V1')
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features   # 512
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


model_optical = build_resnet18(num_classes=NUM_CLASSES).to(DEVICE)
model_sar     = build_resnet18(num_classes=NUM_CLASSES).to(DEVICE)

total_opt  = sum(p.numel() for p in model_optical.parameters())
train_opt_ = sum(p.numel() for p in model_optical.parameters() if p.requires_grad)

print('Model A – ResNet-18 (Optical Only)')
print(f'  Total parameters     : {total_opt:,}')
print(f'  Trainable parameters : {train_opt_:,}')

print('\nModel B – ResNet-18 (SAR Only)')
print(f'  Total parameters     : {total_opt:,}   [same architecture]')

# Sanity check
dummy = torch.randn(2, 3, 64, 64).to(DEVICE)
with torch.no_grad():
    out = model_optical(dummy)
print(f'\n  Output shape (batch=2): {out.shape}  ✅')


In [ ]:
# ── Model C: Feature Fusion Multimodal ResNet-18 ──────────────────────────────

class FusionResNet18(nn.Module):
    """
    Dual-encoder feature fusion model.
    - Optical branch : ResNet-18 (pretrained)
    - SAR branch     : ResNet-18 (pretrained)
    - Features concatenated (512+512=1024) → FC classifier
    Input: (B, 3, 64, 64) optical + (B, 3, 64, 64) SAR
    Output: (B, num_classes)
    """

    def __init__(self, num_classes=10):
        super().__init__()

        # Optical encoder: ResNet-18 without final FC
        resnet_opt     = models.resnet18(weights='IMAGENET1K_V1')
        self.enc_opt   = nn.Sequential(*list(resnet_opt.children())[:-1])  # → (B, 512, 1, 1)

        # SAR encoder: ResNet-18 without final FC
        resnet_sar     = models.resnet18(weights='IMAGENET1K_V1')
        self.enc_sar   = nn.Sequential(*list(resnet_sar.children())[:-1])  # → (B, 512, 1, 1)

        # Fusion classifier: 1024 → 512 → 256 → num_classes
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x_opt, x_sar):
        f_opt = self.enc_opt(x_opt)   # (B, 512, 1, 1)
        f_sar = self.enc_sar(x_sar)   # (B, 512, 1, 1)
        fused = torch.cat([f_opt, f_sar], dim=1)   # (B, 1024, 1, 1)
        return self.classifier(fused)


model_fusion = FusionResNet18(num_classes=NUM_CLASSES).to(DEVICE)

total_f  = sum(p.numel() for p in model_fusion.parameters())
train_f  = sum(p.numel() for p in model_fusion.parameters() if p.requires_grad)
print('Model C – Feature Fusion Multimodal ResNet-18')
print(f'  Total parameters     : {total_f:,}')
print(f'  Trainable parameters : {train_f:,}')
print(f'  Fusion strategy      : Concatenation (512 + 512 = 1024 features)')

dummy_opt = torch.randn(2, 3, 64, 64).to(DEVICE)
dummy_sar = torch.randn(2, 3, 64, 64).to(DEVICE)
with torch.no_grad():
    out_f = model_fusion(dummy_opt, dummy_sar)
print(f'\n  Output shape (batch=2): {out_f.shape}  ✅')


---
## 🏋️ Step 6: Training Functions with Early Stopping

In [ ]:
class EarlyStopping:
    """Patience-based early stopping to prevent overfitting."""
    def __init__(self, patience=10, min_delta=1e-4, restore_best=True):
        self.patience     = patience
        self.min_delta    = min_delta
        self.restore_best = restore_best
        self.counter      = 0
        self.best_loss    = float('inf')
        self.best_weights = None
        self.triggered    = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            if self.restore_best:
                import copy
                self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
                if self.restore_best and self.best_weights:
                    model.load_state_dict(self.best_weights)
        return self.triggered


# ── Training function for single-modality models ───────────────────────────────
def train_model(model, train_loader, val_loader,
                num_epochs=50, lr=5e-4, weight_decay=1e-4,
                patience=10, model_name='model'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6)
    stopper   = EarlyStopping(patience=patience)

    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  []}

    print(f'\n{"="*60}')
    print(f'  Training: {model_name}')
    print(f'  Epochs={num_epochs}, LR={lr}, WD={weight_decay}, Patience={patience}')
    print(f'{"="*60}')

    for epoch in range(1, num_epochs + 1):
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item() * imgs.size(0)
            t_correct += (out.argmax(1) == lbls).sum().item()
            t_total   += imgs.size(0)

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                out  = model(imgs)
                loss = criterion(out, lbls)
                v_loss    += loss.item() * imgs.size(0)
                v_correct += (out.argmax(1) == lbls).sum().item()
                v_total   += imgs.size(0)

        t_l = t_loss / t_total;  t_a = t_correct / t_total * 100
        v_l = v_loss / v_total;  v_a = v_correct / v_total * 100
        history['train_loss'].append(t_l); history['val_loss'].append(v_l)
        history['train_acc'].append(t_a);  history['val_acc'].append(v_a)
        scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Ep {epoch:3d}/{num_epochs} | '
                  f'TrLoss: {t_l:.4f}  TrAcc: {t_a:.1f}% | '
                  f'VaLoss: {v_l:.4f}  VaAcc: {v_a:.1f}%')

        if stopper.step(v_l, model):
            print(f'  ⏹  Early stopping at epoch {epoch} (best val loss: {stopper.best_loss:.4f})')
            break

    print(f'{"="*60}')
    return history


# ── Training function for fusion model ────────────────────────────────────────
def train_fusion_model(model, train_loader, val_loader,
                       num_epochs=50, lr=5e-4, weight_decay=1e-4,
                       patience=10, model_name='Fusion Model'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6)
    stopper   = EarlyStopping(patience=patience)

    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  []}

    print(f'\n{"="*60}')
    print(f'  Training: {model_name}')
    print(f'  Epochs={num_epochs}, LR={lr}, WD={weight_decay}, Patience={patience}')
    print(f'{"="*60}')

    for epoch in range(1, num_epochs + 1):
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for opt_imgs, sar_imgs, lbls in train_loader:
            opt_imgs = opt_imgs.to(DEVICE)
            sar_imgs = sar_imgs.to(DEVICE)
            lbls     = lbls.to(DEVICE)
            optimizer.zero_grad()
            out  = model(opt_imgs, sar_imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item() * opt_imgs.size(0)
            t_correct += (out.argmax(1) == lbls).sum().item()
            t_total   += opt_imgs.size(0)

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for opt_imgs, sar_imgs, lbls in val_loader:
                opt_imgs = opt_imgs.to(DEVICE)
                sar_imgs = sar_imgs.to(DEVICE)
                lbls     = lbls.to(DEVICE)
                out  = model(opt_imgs, sar_imgs)
                loss = criterion(out, lbls)
                v_loss    += loss.item() * opt_imgs.size(0)
                v_correct += (out.argmax(1) == lbls).sum().item()
                v_total   += opt_imgs.size(0)

        t_l = t_loss / t_total;  t_a = t_correct / t_total * 100
        v_l = v_loss / v_total;  v_a = v_correct / v_total * 100
        history['train_loss'].append(t_l); history['val_loss'].append(v_l)
        history['train_acc'].append(t_a);  history['val_acc'].append(v_a)
        scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Ep {epoch:3d}/{num_epochs} | '
                  f'TrLoss: {t_l:.4f}  TrAcc: {t_a:.1f}% | '
                  f'VaLoss: {v_l:.4f}  VaAcc: {v_a:.1f}%')

        if stopper.step(v_l, model):
            print(f'  ⏹  Early stopping at epoch {epoch} (best val loss: {stopper.best_loss:.4f})')
            break

    print(f'{"="*60}')
    return history


---
## 🚀 Step 7: Train Model A – ResNet-18 on Optical Images

In [ ]:
history_optical = train_model(
    model        = model_optical,
    train_loader = train_loader_opt,
    val_loader   = val_loader_opt,
    num_epochs   = 60,
    lr           = 5e-4,
    weight_decay = 1e-4,
    patience     = 12,
    model_name   = 'ResNet-18 (Optical Only)'
)

torch.save(model_optical.state_dict(), 'resnet18_optical_best.pth')
print('💾 Model A saved → resnet18_optical_best.pth')


---
## 🚀 Step 8: Train Model B – ResNet-18 on SAR Images

In [ ]:
history_sar = train_model(
    model        = model_sar,
    train_loader = train_loader_sar,
    val_loader   = val_loader_sar,
    num_epochs   = 60,
    lr           = 5e-4,
    weight_decay = 1e-4,
    patience     = 12,
    model_name   = 'ResNet-18 (SAR Only)'
)

torch.save(model_sar.state_dict(), 'resnet18_sar_best.pth')
print('💾 Model B saved → resnet18_sar_best.pth')


---
## 🚀 Step 9: Train Model C – Feature Fusion (SAR + Optical)

In [ ]:
history_fusion = train_fusion_model(
    model        = model_fusion,
    train_loader = train_loader_mm,
    val_loader   = val_loader_mm,
    num_epochs   = 60,
    lr           = 5e-4,
    weight_decay = 1e-4,
    patience     = 12,
    model_name   = 'Feature Fusion ResNet-18 (Optical + SAR)'
)

torch.save(model_fusion.state_dict(), 'resnet18_fusion_best.pth')
print('💾 Model C saved → resnet18_fusion_best.pth')


---
## 📈 Step 10: Learning Curves – All Three Models

In [ ]:
def plot_learning_curves(history, title, color='steelblue'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], '-o', markersize=3,
             color=color, label='Train Loss')
    ax1.plot(epochs, history['val_loss'],   '-s', markersize=3,
             color='crimson', label='Val Loss')
    ax1.set_title(f'{title} – Loss Curve', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, history['train_acc'], '-o', markersize=3,
             color=color, label='Train Acc')
    ax2.plot(epochs, history['val_acc'],   '-s', markersize=3,
             color='crimson', label='Val Acc')
    ax2.set_title(f'{title} – Accuracy Curve', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_ylim(0, 105)
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    fname = title.replace(' ', '_').lower().replace('(', '').replace(')', '') + '_curves.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


plot_learning_curves(history_optical, 'ResNet-18 – Optical Only',   color='steelblue')
plot_learning_curves(history_sar,     'ResNet-18 – SAR Only',        color='darkorange')
plot_learning_curves(history_fusion,  'Fusion ResNet-18 – SAR+Optical', color='seagreen')
print('✅ Learning curves plotted for all three models')


---
## 📊 Step 11: Test Set Evaluation – All Three Models

In [ ]:
def evaluate_model(model, loader, model_name, fusion=False):
    """Evaluate model on test set, return metrics dict."""
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        if fusion:
            for opt_imgs, sar_imgs, lbls in loader:
                opt_imgs = opt_imgs.to(DEVICE)
                sar_imgs = sar_imgs.to(DEVICE)
                out   = model(opt_imgs, sar_imgs)
                preds = out.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_true.extend(lbls.numpy())
        else:
            for imgs, lbls in loader:
                imgs  = imgs.to(DEVICE)
                out   = model(imgs)
                preds = out.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_true.extend(lbls.numpy())

    acc = accuracy_score(all_true, all_preds) * 100
    f1  = f1_score(all_true, all_preds, average='macro') * 100

    print(f'\n{"="*60}')
    print(f'  {model_name} – Test Set Results')
    print(f'{"="*60}')
    print(f'  Accuracy (Overall) : {acc:.2f}%')
    print(f'  F1 Score (macro)   : {f1:.2f}%')
    print(f'{"="*60}')
    print('\n  Per-Class Classification Report:')
    print(classification_report(all_true, all_preds,
                                target_names=CLASS_NAMES, digits=3))

    return {
        'true_labels': np.array(all_true),
        'pred_labels': np.array(all_preds),
        'accuracy'   : acc,
        'f1'         : f1
    }


results_optical = evaluate_model(model_optical, test_loader_opt,
                                  'ResNet-18 (Optical Only)')
results_sar     = evaluate_model(model_sar,     test_loader_sar,
                                  'ResNet-18 (SAR Only)')
results_fusion  = evaluate_model(model_fusion,  test_loader_mm,
                                  'Fusion ResNet-18 (SAR + Optical)', fusion=True)


---
## 🔥 Step 12: Confusion Matrices

In [ ]:
def plot_confusion_matrix(true_labels, pred_labels, class_names, title):
    cm      = confusion_matrix(true_labels, pred_labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=axes[0])
    axes[0].set_title(f'{title}\nConfusion Matrix (Counts)', fontweight='bold')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    axes[0].tick_params(axis='x', rotation=40, labelsize=8)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, vmin=0, vmax=1, ax=axes[1])
    axes[1].set_title(f'{title}\nNormalized Confusion Matrix', fontweight='bold')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    axes[1].tick_params(axis='x', rotation=40, labelsize=8)

    plt.tight_layout()
    fname = title.replace(' ', '_').lower()[:30] + '_cm.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


plot_confusion_matrix(results_optical['true_labels'], results_optical['pred_labels'],
                      CLASS_NAMES, 'ResNet-18 Optical Only')
plot_confusion_matrix(results_sar['true_labels'],     results_sar['pred_labels'],
                      CLASS_NAMES, 'ResNet-18 SAR Only')
plot_confusion_matrix(results_fusion['true_labels'],  results_fusion['pred_labels'],
                      CLASS_NAMES, 'Fusion ResNet-18 SAR+Optical')
print('✅ Confusion matrices saved')


---
## 📊 Step 13: Final Model Comparison – SAR vs Optical vs Fusion

In [ ]:
import pandas as pd

# ── Extract metrics ─────────────────────────────────────────────────────────────
models_list = ['ResNet-18\nOptical Only', 'ResNet-18\nSAR Only',
               'Fusion ResNet-18\nSAR + Optical']
models_short = ['Optical Only', 'SAR Only', 'Fusion (SAR+Optical)']

acc_opt    = results_optical['accuracy']
acc_sar    = results_sar['accuracy']
acc_fusion = results_fusion['accuracy']
f1_opt     = results_optical['f1']
f1_sar     = results_sar['f1']
f1_fusion  = results_fusion['f1']

accuracies = [acc_opt, acc_sar, acc_fusion]
f1_scores  = [f1_opt, f1_sar, f1_fusion]

# ── Comparison Table ────────────────────────────────────────────────────────────
comparison_df = pd.DataFrame({
    'Model'         : models_short,
    'Input Modality': ['Optical (RGB)', 'SAR (NIR/SWIR)', 'SAR + Optical'],
    'Architecture'  : ['ResNet-18', 'ResNet-18', 'Dual ResNet-18 Fusion'],
    'Accuracy (%)'  : [f'{acc_opt:.2f}', f'{acc_sar:.2f}', f'{acc_fusion:.2f}'],
    'F1-Score (%)'  : [f'{f1_opt:.2f}',  f'{f1_sar:.2f}',  f'{f1_fusion:.2f}'],
    'Parameters'    : ['~11.2M', '~11.2M', '~22.5M'],
})

print('='*90)
print('        MULTIMODAL LAND CLASSIFICATION – FINAL COMPARISON')
print('='*90)
print(comparison_df.to_string(index=False))
print('='*90)

best_idx = np.argmax(accuracies)
print(f'\n🏆 BEST MODEL: {models_short[best_idx]}  → Accuracy: {accuracies[best_idx]:.2f}%')


In [ ]:
# ── Comparison Visualizations ──────────────────────────────────────────────────
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
x = np.arange(len(models_short))
w = 0.35

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Multimodal Remote Sensing Land Classification – Model Comparison\n'
             'EuroSAT Dataset | 100 Patches per Class | ResNet-18',
             fontsize=13, fontweight='bold', y=1.02)

# Accuracy bar chart
ax1 = axes[0]
bars1 = ax1.bar(models_short, accuracies, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax1.set_title('Test Set Accuracy', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 110)
ax1.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars1, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.tick_params(axis='x', labelsize=9)

# F1-Score bar chart
ax2 = axes[1]
bars2 = ax2.bar(models_short, f1_scores, color=colors, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('F1-Score (%)', fontsize=12, fontweight='bold')
ax2.set_title('Test Set F1-Score (Macro)', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 110)
ax2.grid(axis='y', alpha=0.3)
for bar, f1 in zip(bars2, f1_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{f1:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.tick_params(axis='x', labelsize=9)

# Grouped Accuracy vs F1
ax3 = axes[2]
bars3a = ax3.bar(x - w/2, accuracies, w, label='Accuracy',  color='steelblue', edgecolor='black')
bars3b = ax3.bar(x + w/2, f1_scores,  w, label='F1-Score',  color='darkorange', edgecolor='black')
ax3.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax3.set_title('Accuracy vs F1-Score Grouped', fontsize=12, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(models_short, fontsize=9)
ax3.set_ylim(0, 115)
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)
for bar in list(bars3a) + list(bars3b):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('multimodal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison chart saved → multimodal_comparison.png')


---
## 🔍 Step 14: Misclassified Patches – Best Model

In [ ]:
# ── Visualize misclassified test patches from the best model ──────────────────
best_acc = max(acc_opt, acc_sar, acc_fusion)

if best_acc == acc_fusion:
    best_model_name = 'Fusion ResNet-18 (SAR+Optical)'
    wrong_images_opt, wrong_images_sar, wrong_true, wrong_pred = [], [], [], []
    model_fusion.eval()
    for i, (img_opt, img_sar, true_lbl) in enumerate(
            zip(test_opt, test_sar, test_lbl)):
        o = img_opt if img_opt.mode == 'RGB' else img_opt.convert('RGB')
        s = img_sar if img_sar.mode == 'RGB' else img_sar.convert('RGB')
        t_o = val_transform(o).unsqueeze(0).to(DEVICE)
        t_s = val_transform(s).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred = model_fusion(t_o, t_s).argmax(dim=1).item()
        if pred != true_lbl:
            wrong_images_opt.append(o)
            wrong_true.append(true_lbl)
            wrong_pred.append(pred)
        if len(wrong_images_opt) == 10: break

    n_show = len(wrong_images_opt)
    fig, axes = plt.subplots(2, n_show, figsize=(n_show*3, 6))
    fig.suptitle(f'Misclassified Test Patches – {best_model_name}',
                 fontsize=13, fontweight='bold')
    for i in range(n_show):
        axes[0, i].imshow(wrong_images_opt[i])
        axes[0, i].set_title(f'True: {CLASS_NAMES[wrong_true[i]]}\n'
                              f'Pred: {CLASS_NAMES[wrong_pred[i]]}',
                              fontsize=7, color='red')
        axes[0, i].axis('off')
    plt.tight_layout()

else:
    chosen_model  = model_optical if best_acc == acc_opt else model_sar
    chosen_loader = test_opt      if best_acc == acc_opt else test_sar
    best_model_name = 'Optical Only' if best_acc == acc_opt else 'SAR Only'
    wrong_images, wrong_true, wrong_pred = [], [], []
    chosen_model.eval()
    for img, true_lbl in zip(chosen_loader, test_lbl):
        rgb = img if img.mode == 'RGB' else img.convert('RGB')
        t   = val_transform(rgb).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred = chosen_model(t).argmax(dim=1).item()
        if pred != true_lbl:
            wrong_images.append(rgb)
            wrong_true.append(true_lbl)
            wrong_pred.append(pred)
        if len(wrong_images) == 12: break

    n_show = min(12, len(wrong_images))
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    fig.suptitle(f'Misclassified Test Patches – {best_model_name}',
                 fontsize=13, fontweight='bold')
    for i, ax in enumerate(axes.flatten()):
        if i < n_show:
            ax.imshow(wrong_images[i])
            ax.set_title(f'True: {CLASS_NAMES[wrong_true[i]]}\n'
                         f'Pred: {CLASS_NAMES[wrong_pred[i]]}',
                         fontsize=7, color='red')
        ax.axis('off')

plt.tight_layout()
plt.savefig('misclassified_patches.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Misclassified patches visualized for best model: {best_model_name}')


---
## 📝 Step 15: Final Summary & Discussion

In [ ]:
print('='*70)
print('    MULTIMODAL REMOTE SENSING LAND CLASSIFICATION – FINAL REPORT')
print('='*70)
print(f"""
  Dataset        : EuroSAT (Sentinel-2 patches)
  Patches used   : 100 per class × 10 classes = 1000 total
  Split          : 60% Train / 20% Val / 20% Test (stratified)
  Device         : {DEVICE}

  Modalities:
    Optical (RGB) : Bands B4, B3, B2 — True color RGB imagery
    SAR-simulated : Bands B8, B11, B12 — NIR + SWIR texture features
""")
print('  Model Performance on Test Set:')
print(comparison_df.to_string(index=False))
print(f"""
  Key Takeaways:
  ─────────────────────────────────────────────────────────────
  1. MULTIMODAL ADVANTAGE: The Fusion model combines complementary
     information from optical (color/spectral) and SAR (structural/
     texture) features, typically achieving higher accuracy than
     either single modality alone.

  2. SAR vs OPTICAL: Optical imagery generally performs better on
     vegetated classes (Forest, HerbaceousVegetation) due to rich
     spectral signatures. SAR-simulated bands better distinguish
     structural classes (Industrial, Residential, Highway).

  3. FEATURE FUSION: Concatenating 512-dim embeddings from two
     ResNet-18 encoders (1024-dim fused vector) allows the model to
     jointly learn discriminative representations from both modalities.

  4. TRANSFER LEARNING: ResNet-18 pretrained on ImageNet provides
     strong feature extraction even for remote sensing data, reducing
     the need for large training sets.

  5. DIFFICULT CLASSES: AnnualCrop ↔ PermanentCrop and
     HerbaceousVegetation ↔ Pasture remain challenging due to
     similar spectral and structural signatures.

  6. FUTURE WORK: True SAR (Sentinel-1 C-band) data with EuroSAT-SAR
     dataset would further strengthen the multimodal pipeline.
""")
print('='*70)


---
## 📚 References

1. **Helber et al. (2019)** – *EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification*. IEEE JSTARS.
2. **He et al. (2016)** – *Deep Residual Learning for Image Recognition*. CVPR 2016.
3. **EuroSAT Dataset** – https://github.com/phelber/EuroSAT
4. **TorchGeo** – https://torchgeo.readthedocs.io
5. **PyTorch Documentation** – https://pytorch.org/docs
6. **scikit-learn** – https://scikit-learn.org
7. **Schmitt & Zhu (2016)** – *Data Fusion and Remote Sensing: An Ever-Growing Relationship*. IEEE Geoscience and Remote Sensing Magazine.

---
*Notebook prepared for AIML Lab Project | B.Tech 4th Semester 2025*
